
# Set F — Synapses & Two‑Neuron Communication (LEGO–Colab, fixed v4)

**Final (Option 1 style)** — 2026-03-09  \
**Author:** Satish Nair

**Run order:** F0 ▶ F1 ▶ … ▶ F9. If things desync: *Runtime ▶ Restart & run all*.  
This version uses **NetStim** (built‑in artificial cell) instead of `VecStim` to maximize Colab compatibility.

---
## Table of Contents
- [F0 — Starter](#f0)
- [F1 — Two neurons + AMPA‑like synapse](#f1)
- [F2 — Inhibitory (GABA\_A‑like)](#f2)
- [F3 — EPSP vs IPSP (peaks & latencies)](#f3)
- [F4 — Exercise: what makes a synapse excitatory? (E\_rev sweep)](#f4)
- [F5 — Temporal summation (two spikes, variable ISI)](#f5)
- [F6 — Spatial summation (proximal vs distal)](#f6)
- [F7 — Feedforward inhibition (E then I)](#f7)
- [F8 — Analysis helpers (latency, peak, area)](#f8)
- [F9 — Playground](#f9)
- [Reflection](#reflection)
---


## F0 — Starter (run once) <a id='f0'></a>

In [ ]:

!pip -q install neuron==8.2.4
try:
    from neuron import h, gui
except Exception:
    from neuron import h
from neuron.units import ms, mV
import numpy as np
import matplotlib.pyplot as plt
h.load_file('stdrun.hoc')

# Bootstrap section so HOC has an access context
s0 = h.Section(name='bootstrap')

# Plot helper
def plot_vm(t_vec, traces, labels, title='', figsize=(7,3.3)):
    plt.figure(figsize=figsize)
    for tr,lb in zip(traces, labels):
        plt.plot(t_vec, tr, label=lb)
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(title)
    plt.grid(True, alpha=0.3);
    if len(labels)>1: plt.legend()
    plt.tight_layout(); plt.show()

# Global time recorder (persistent)
t = h.Vector(); t.record(h._ref_t)

# Recorder helper returns a persistent Vector
def mk_rec(sec):
    v = h.Vector(); v.record(sec(0.5)._ref_v); return v

print('F0 ready. Proceed to F1.')


## F1 — Two neurons + excitatory synapse (AMPA‑like) <a id='f1'></a>

In [ ]:

somaA = h.Section(name='somaA'); somaA.L=somaA.diam=20; somaA.Ra=100; somaA.cm=1
somaA.insert('hh')
somaB = h.Section(name='somaB'); somaB.L=somaB.diam=20; somaB.Ra=100; somaB.cm=1
somaB.insert('hh')

iclampA = h.IClamp(somaA(0.5)); iclampA.delay=5; iclampA.dur=2; iclampA.amp=0.4
syn = h.ExpSyn(somaB(0.5)); syn.tau=2.0; syn.e=0.0  # AMPA-like
nc = h.NetCon(somaA(0.5)._ref_v, syn, sec=somaA)
nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.005
vA = mk_rec(somaA); vB = mk_rec(somaB)
h.finitialize(-65*mV); h.continuerun(30*ms)
plot_vm(t, [vA, vB], ['Neuron A','Neuron B'], title='F1: A→B excitatory (AMPA-like)')


> **Systems callout — Point‑to‑point signaling**  
- Presynaptic spike → neurotransmitter release → postsynaptic conductance transient (ExpSyn).
- Driving force: I_syn = g(t)·(E_syn − V). If E_syn > V_rest, EPSP; if E_syn < V_rest, small or hyperpolarizing.

**Try with Copilot**  
> Predict the EPSP peak if you doubled `nc.weight[0]` and explain limits (e.g., saturation, spike triggering).

## F2 — Switch to inhibitory synapse (GABA_A‑like) <a id='f2'></a>

In [ ]:

syn.e = -75.0  # GABA_A-like reversal
syn.tau = 8.0
nc.weight[0] = 0.01
h.finitialize(-65*mV); h.continuerun(40*ms)
plot_vm(t, [vA, vB], ['Neuron A','Neuron B'], title='F2: A→B inhibitory (GABA_A-like)')


> **Systems callout — Sign from reversal**  
- If E_syn << V_rest, activation drives V toward E_syn → IPSP.
- Longer τ slows kinetics; temporal spread influences summation.

**Try with Copilot**  
> For E_syn = −75 mV, why might an IPSP slightly depolarize if the cell is far below −75 mV?

## F3 — EPSP vs IPSP contrast (peaks & latencies) <a id='f3'></a>

In [ ]:

# compare EPSP vs IPSP
import numpy as np

def run_trial(e_rev, tau, w, ttl):
    syn.e=e_rev; syn.tau=tau; nc.weight[0]=w
    h.finitialize(-65*mV); h.continuerun(35*ms)
    vnp = np.array(vB); tnp=np.array(t)
    peak=vnp.max(); tpk=tnp[vnp.argmax()]
    plot_vm(t, [vB], ['Neuron B'], title=f'{ttl} | peak={peak:.1f} mV @ {tpk:.2f} ms')
    return peak, tpk
p1 = run_trial(0.0, 2.0, 0.005, 'F3: EPSP')
p2 = run_trial(-75.0, 8.0, 0.01, 'F3: IPSP')
print('EPSP:', p1, 'IPSP:', p2)


> **Systems callout — Kinetics & amplitude**  
- EPSPs (fast, small τ) peak earlier than IPSPs (slower, larger τ).
- Peak and latency also depend on `weight` and membrane time constant.

**Try with Copilot**  
> Halve `τ` and `weight` together; predict how peak and timing change relative to baseline.

## F4 — *Exercise*: excitatory vs inhibitory? (Reversal sweep) <a id='f4'></a>

In [ ]:

# Reversal potential sweep
def reversal_sweep(E_values=[-85,-75,-65,-55,-45,-35,-25,-15,-5,5,10], tau=4.0, w=0.006):
    peaks=[]
    for e in E_values:
        syn.e = e; syn.tau=tau; nc.weight[0]=w
        h.finitialize(-65*mV); h.continuerun(35*ms)
        vnp=np.array(vB); tnp=np.array(t)
        peaks.append((e, vnp.max()))
        plt.figure(figsize=(5.4,2.6))
        plt.plot(tnp, vnp, 'k'); plt.axhline(-65, color='gray', ls='--', lw=0.8)
        plt.title(f'F4: E_syn={e} mV'); plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
    return peaks
peaks = reversal_sweep()
print('Peak Vm by E_syn (mV):', peaks)


> **Systems callout — E_syn vs V_rest**  
- Synapse is functionally excitatory when E_syn above operating V; inhibitory when below.
- Sweep reveals the sign flip and sensitivity around rest.

**Try with Copilot**  
> Identify the `E_syn` at which PSP switches sign; explain using I = g(t)(E_syn − V).

## F5 — Temporal summation (two spikes, variable ISI) <a id='f5'></a>

In [ ]:

# two presynaptic events using NetStim (built‑in artificial cell)
def temporal_sum(isi=5.0, e_rev=0.0, tau=2.0, w=0.005, start=5.0):
    syn.e = e_rev; syn.tau = tau
    ns = h.NetStim()
    ns.number = 2
    ns.start = start
    ns.interval = isi
    ns.noise = 0
    nc_ns = h.NetCon(ns, syn)
    nc_ns.weight[0] = w
    h.finitialize(-65*mV); h.continuerun((start+isi+15)*ms)
    plot_vm(t, [vB], ['Neuron B'], title=f'F5: Temporal summation (ISI={isi} ms)')
for isi in [2.0, 5.0, 20.0]:
    temporal_sum(isi=isi)


> **Systems callout — ISI and PSP overlap**  
- Shorter ISI → greater summation if τ_syn and τ_membrane allow overlap.
- Relate to postsynaptic filtering: τ_membrane integrates presynaptic events.

**Try with Copilot**  
> Estimate the effective summation window from your τ_syn and τ_membrane.

## F6 — Spatial summation (proximal vs distal dendritic synapses) <a id='f6'></a>

In [ ]:

# add dendrite to B
try:
    dendB
except NameError:
    dendB = h.Section(name='dendB'); dendB.L=300; dendB.diam=1.5; dendB.Ra=100; dendB.cm=1; dendB.insert('pas'); dendB.connect(somaB(1.0))

synP = h.ExpSyn(dendB(0.2)); synP.tau=2.0; synP.e=0.0
synD = h.ExpSyn(dendB(0.8)); synD.tau=2.0; synD.e=0.0
# Two one-shot NetStims starting at the same time (6 ms)
nsP = h.NetStim(); nsP.number=1; nsP.start=6.0; nsP.noise=0
nsD = h.NetStim(); nsD.number=1; nsD.start=6.0; nsD.noise=0
ncP = h.NetCon(nsP, synP); ncP.weight[0]=0.005
ncD = h.NetCon(nsD, synD); ncD.weight[0]=0.005

v_soma = mk_rec(somaB)
v_prox = h.Vector().record(dendB(0.2)._ref_v)
v_dist = h.Vector().record(dendB(0.8)._ref_v)

h.finitialize(-65*mV); h.continuerun(25*ms)
plt.figure(figsize=(7,3.3))
plt.plot(t, v_prox, label='dend(0.2)')
plt.plot(t, v_dist, label='dend(0.8)')
plt.plot(t, v_soma, label='soma', color='k')
plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.title('F6: Spatial summation & attenuation')
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()


> **Systems callout — Electrotonic distance**  
- Distal inputs attenuate more at soma; proximal inputs dominate summation.
- λ and dendritic geometry shape amplitude and timing at soma.

**Try with Copilot**  
> Reduce dendritic diameter and predict changes in soma PSPs from proximal vs distal sites.

## F7 — Feedforward inhibition (E then I) <a id='f7'></a>

In [ ]:

synE2 = h.ExpSyn(somaB(0.5)); synE2.tau=2.0; synE2.e=0.0
synI2 = h.ExpSyn(somaB(0.5)); synI2.tau=8.0; synI2.e=-75.0
nsE = h.NetStim(); nsE.number=1; nsE.start=6.0; nsE.noise=0
nsI = h.NetStim(); nsI.number=1; nsI.start=8.0; nsI.noise=0
ncE2 = h.NetCon(nsE, synE2); ncE2.weight[0]=0.006
ncI2 = h.NetCon(nsI, synI2); ncI2.weight[0]=0.01
v_ff = mk_rec(somaB)
h.finitialize(-65*mV); h.continuerun(25*ms)
plot_vm(t, [v_ff], ['Neuron B'], title='F7: Feedforward inhibition (E→I)')


> **Systems callout — Timing balance**  
- E followed by I gates spike generation; delays tune window of excitability.
- Common motif for shaping temporal precision in circuits.

**Try with Copilot**  
> Sweep the E→I delay and report the PSP peak; identify delay that maximally suppresses spiking.

## F8 — Analysis helpers (latency, peak, area) <a id='f8'></a>

In [ ]:

# metrics
import numpy as np

def psp_metrics(tvec, vtrace, baseline=(0,5)):
    tnp=np.array(tvec); vnp=np.array(vtrace)
    b=(tnp>=baseline[0])&(tnp<=baseline[1]); v0=vnp[b].mean() if b.any() else vnp[0]
    peak=vnp.max(); tpk=tnp[vnp.argmax()]
    area=np.trapz(vnp-v0, tnp)
    return {'V0':v0,'Vpeak':peak,'tpeak_ms':tpk,'Area_mV_ms':area}

print('F8 metrics (last trace on vB):', psp_metrics(t, vB))


> **Systems callout — Compact PSP descriptors**  
- Latency to peak, peak amplitude, and area summarize kinetics and strength.
- Use for parameter sweeps (e.g., `E_syn`, `τ`, `weight`) and plotting curves.

**Try with Copilot**  
> Automate a sweep of `τ` and plot Vpeak and area vs τ for EPSPs.

## F9 — Playground (tweak synaptic parameters and rerun) <a id='f9'></a>

In [ ]:

syn.e = 0.0; syn.tau=2.0; nc.weight[0]=0.008; nc.delay=0.5
h.finitialize(-65*mV); h.continuerun(30*ms)
plot_vm(t, [vA, vB], ['Neuron A','Neuron B'], title='F9: Playground run')


## Reflection <a id='reflection'></a>
- How do E_syn and τ jointly control PSP amplitude and timing relative to membrane τ?  
- Under what conditions can inhibitory input produce a depolarizing PSP and still be functionally inhibitory?  
- How do temporal (ISI) and spatial (electrotonic distance) factors interact to shape postsynaptic integration?